# Phase 2C4 on Colab — the "filtered" condition (denoised hard negatives)

**What this is:** the one strategy the campaign left untested. C4 = C2's BM25 ranking with two denoisers taken from the audit's own separability cut: drop every candidate whose token-Jaccard with the anchor is ≥ **0.40**, then skip the first **10** survivors. Everything else — exclusion pass, corpus filter, shared anchor/positive stream, margin 0.10, cap — is identical to C1–C3. Pre-registration: `sanitycheck.md` §9.4 (commit `847a698`), written before any C4 run existed.

**Pre-registered expectation:** F1(C4) ≈ F1(C1). The informative outcomes: C4 ≈ C1 (denoising rescues hard mining to the clean baseline), C1 < C4 (denoised hard negatives add real signal — the RocketQA best case), C4 ≈ C2 (the poison is semantic, not lexical — lexical denoising cannot fix it).

**MANDATORY before any C4 number is compared to anything:** a fresh 50-pair blind audit of the C4 triples (§5 below), gated at the C2 floor (42%). The audit writes to its own directory (`artifacts/audit_c4/`) and never touches the scored Phase-2 sheet.

**Order of operations (each section is a designed stop):**
1. §2 checkout + runtime + artifacts pull (`--light`: no checkpoints)
2. §3 re-mine under `phase01-v7` — backup → re-mine all four strategies → **assert C1–C3 bit-identical** (the audit key survival check) → C4 is the only new bytes
3. §4 hardness diagnostics: is C4 hard-but-clean?
4. §5 **AUDIT GATE** (manual labelling — stop here until scored and passed)
5. §6 train + evaluate C4 × seeds 13–18 (n = 6, symmetric with C0–C3), push run dirs
6. §7 consolidator: report table + confusion board; §8 GitHub push; §9 reading the results

Budget: mining ~10 min CPU + one smoke ~3 min; training ~2–3 h per seed on a T4 (six seeds ≈ one VM-day, or split 3+3 across two VMs like Phase 2V2).

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ksu-gitreaper807/EmbedEd.git'
REPO_REF = 'arena/01a0e714-embeded'  # current Arena work branch (D1 gate rule, Phase 2 runner, Phase 3 runner); use main once merged
ROOT = '/content/EmbedEd'

def _git(*args, check=True):
    p = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    if check and p.returncode:
        raise SystemExit(f"git {' '.join(args)} exited {p.returncode}\n{p.stdout}{p.stderr}")
    return p.stdout.strip()

if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, ROOT], check=True)
else:
    _git('fetch', 'origin', REPO_REF)
    ahead = _git('rev-list', '--count', 'FETCH_HEAD..HEAD', check=False)
    if ahead not in ('', '0'):
        raise SystemExit(
            f'{ahead} local commit(s) on this VM are not on origin/{REPO_REF}. Refusing to move the '
            'branch and orphan them: run notebook section 8 to push them (or `git rebase FETCH_HEAD` '
            'and push by hand), then re-run this cell.')
    _git('checkout', '-B', REPO_REF, 'FETCH_HEAD')
%cd /content/EmbedEd
print('Checked out:', _git('rev-parse', '--short', 'HEAD'))


## 1. Configure caches, install pinned dependencies, verify GPU, pull artifacts

Same environment as Phase 2V2. Mining and the diagnostics are CPU-only; the T4 is for training.

In [ ]:
import os, sys
ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ['EMBEDED_ARTIFACTS'] = ART
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION', 'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2 matplotlib==3.9.4
import torch, transformers, datasets, sentence_transformers, sklearn
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} | transformers={transformers.__version__} | datasets={datasets.__version__} | sklearn={sklearn.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('GPU is required. Select a Colab T4 runtime before continuing.')
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu.upper(), f'Expected the measured T4 configuration; got {gpu}. Re-measure throughput before using the frozen budget.'
from scripts.hf_artifacts import main as hf_main
# --light: trainers need fragments/triples/mining_summary, NOT the ~4.5 GB of checkpoints
hf_main(['pull', '--if-configured', '--light'])

## 2. Preflight: frozen artifacts

The G1 gate is **not** re-run. One expected difference from Phase 2V2: `mining_summary.json` still says `phase01-v6` at this point — the §3 re-mine (sanctioned, version-bumped) updates it. Everything else must look exactly like a Phase 2 VM.

In [ ]:
from pathlib import Path
from embeded import settings as S
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy'), S.artifact('corpus_emb.meta.json')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1', 'C2', 'C3')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing artifacts; re-pull (cell above) before continuing:\n' + '\n'.join(missing))
print('Artifacts present. Checkout VERSION =', S.VERSION)
import json as _json
_summary = S.ARTIFACTS / 'mining_summary.json'
if _summary.exists():
    _v = (_json.loads(_summary.read_text()).get('_run') or {}).get('version')
    print('mining_summary.json _run.version =', _v)
    if _v == S.VERSION:
        print('already v7 - section 3 will print [cache] and skip (delete triples_C*.jsonl to force)')
    elif _v == 'phase01-v6':
        print('v6 artifacts: EXPECTED. Section 3 performs the sanctioned version-bump re-mine.')
    else:
        raise SystemExit(f'unexpected mining version {_v!r} - stop and check with the notebook author.')
else:
    print('mining_summary.json absent - section 3 mines from scratch (fine).')

## 3. The sanctioned v7 re-mine — with the audit-key survival check

Why all four strategies and not just C4: `embeded.train`/`embeded.evaluate` **refuse** artifacts whose `mining_summary.json` version differs from the checkout's `VERSION`. The v7 bump (C4's constants live in `settings.py`) therefore requires one re-mine so the summary says `phase01-v7`.

Why that is safe: mining is deterministic given the frozen seeds and the frozen Phase 1 artifacts — the v7 change adds the `filtered` strategy and touches no C1–C3 code path. So the protocol below **backs up the v6 triples, re-mines, and asserts the new C1–C3 files are bit-identical to the backups**. If that assertion passes, the blind audit's key still matches the triples on disk and every Phase-2 label survives. If it ever fails: STOP — restore the `.v6bak` files and re-label the audit before trusting anything.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
from embeded import settings as S
import embeded.negatives as NG

def sha(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()

# --- staleness guard: this cell needs the C4-era code; fail BEFORE backing up ---
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
print('checkout HEAD:', head, '| VERSION:', S.VERSION)
if not (hasattr(NG, 'clean_candidates_filtered') and 'filtered' in NG.COND
        and hasattr(S, 'C4_JACCARD_MAX')):
    raise SystemExit('STALE CHECKOUT: this clone predates the C4 strategy (commit 56822ba). '
                     'Re-run the checkout cell (section 0), then re-run THIS cell.')
print('this cell is the f4598cd+ version (streams output, tees a log). If you do not see '
      "'checkout HEAD:' above, you are running a stale copy of this cell - reload the "
      'notebook from GitHub and paste this cell over the old one.')

conds = ('C1', 'C2', 'C3')
before = {}
for c in conds:
    t_path = S.artifact(f'triples_{c}.jsonl')
    bak = Path(str(t_path) + '.v6bak')
    if t_path.exists() and not bak.exists():   # first run: keep the v6 original
        shutil.copy2(t_path, bak)
    if bak.exists():
        before[c] = sha(bak)                   # the assertion is against the ORIGINAL
        print(f'backup on file: {bak.name} ({before[c][:12]}...)')

summary_path = S.ARTIFACTS / 'mining_summary.json'
run = ((json.loads(summary_path.read_text()).get('_run') or {})
       if summary_path.exists() else {})
if run.get('version') == S.VERSION and 'filtered' in run.get('strategies', []):
    print('[cache] mining_summary.json is already v7 with filtered - skipping the re-mine.')
else:
    print('mining all four strategies (BM25/filtered are CPU; ~10 min total)...', flush=True)
    # -u + line-streaming + tee: progress and any traceback ALWAYS reach the cell,
    # and a full copy lands in /content/mining_c4_log.txt (outside ART: never pushed)
    log_path = Path('/content/mining_c4_log.txt')
    with open(log_path, 'w') as log:
        child = subprocess.Popen([sys.executable, '-u', '-m', 'embeded.negatives',
                                  '--strategies', 'random,bm25,semantic,filtered', '--force'],
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1)
        for line in child.stdout:
            log.write(line)
            print(line, end='', flush=True)
        rc = child.wait()
    if rc:
        raise SystemExit(f'mining exited {rc}. Full log: {log_path} - print it with:\n'
                         f'  !tail -n 80 {log_path}\n'
                         'and paste it verbatim into the session. A ValueError/AttributeError '
                         'mentioning "filtered" means the checkout was stale: re-run the '
                         'checkout cell, then this one.')

changed = [c for c in conds if c in before and sha(S.artifact(f'triples_{c}.jsonl')) != before[c]]
if changed:
    raise SystemExit(
        f'RE-MINE CHANGED {changed}: the determinism assumption failed. The blind audit labels are now '
        f'orphaned. Restore:  for c in {" ".join(conds)}: mv artifacts/triples_$c.jsonl.v6bak artifacts/triples_$c.jsonl  '
        '(keep the new C4 file), then re-label the audit before any training.')
if before:
    print('C1-C3 bit-identical to the v6 backups: audit key + labels survive.')

summary = json.loads(summary_path.read_text())
assert (summary.get('_run') or {}).get('version') == S.VERSION, 'summary not v7?'
assert 'filtered' in (summary.get('_run') or {}).get('strategies', []), 'filtered missing from summary'
c4 = summary['C4']
print('C4:', c4['triples'], 'triples | stats:', c4['stats'])
print('vs C2:', summary['C2']['triples'], 'triples |', summary['C2']['stats'])
print('(jaccard_filtered + head_skipped is C4\'s denoiser at work; padded > 0 on many anchors = C4 drifting toward C1)')

## 4. Hardness diagnostics — is C4 hard-but-clean?

The diagnostics auto-include C4 now that `triples_C4.jsonl` exists. Read the C4 row against C1 (clean-easy) and C2 (poisoned-hard): C4 should sit **between** them on mean cosine similarity to the anchor — meaningfully harder than C1, but without C2's contamination mass. Paste this output back with the campaign results.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
p = subprocess.run([sys.executable, '-u', '-m', 'scripts.hardness_diagnostics', '--examples', '2'])
if p.returncode: raise SystemExit(f'hardness_diagnostics exited {p.returncode} - reason printed above.')

## 4.5 Smoke test the runner on this VM

~3 minutes on a T4; writes to its own `runs/smoke_*` directory. It proves the C4 path end-to-end (triples present, trainer accepts the new condition, checkpoint + evaluate work) before you spend the budget.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('runner modules missing from this checkout - check REPO_REF in the checkout cell.')
p = subprocess.run([sys.executable, '-u', '-m', 'embeded.train', '--condition', 'C4', '--seed', '13', '--smoke'])
if p.returncode:
    raise SystemExit(f'smoke run exited {p.returncode} - the reason is printed above. Do not start the campaign until this passes.')

## 5. THE AUDIT GATE — 50-pair blind audit of C4 (MANDATORY, before any training)

§9.4's gate: **no C4 result may be compared to anything until this audit is scored and passed.** The sheet is generated blind (the key stays closed), lives in `artifacts/audit_c4/` — the scored Phase-2 sheet in `artifacts/audit/` is untouched — and asks the same question as the Phase-2 audit: of the mined negatives, how many are actually clones (false negatives)?

**Gate:** C4's audited clone rate must be **≤ 42%** (C2's measured floor) — the pre-registered target is ≈ 0–2% (C1's band). If the gate fails, C4 is reported as "denoising attempted, contamination persists" and the F1 comparison is footnoted, not headlined. Mining, smoke and even training may run, but no comparison gets written down first.

Run the cell, then **stop**: download `audit_pairs.md` from the file browser, label offline, come back to §5.5.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['EMBEDED_AUDIT_DIR'] = ART + '/audit_c4'   # never touch the scored Phase-2 sheet
p = subprocess.run([sys.executable, '-u', '-m', 'scripts.audit_sample', 'make', '--conditions', 'C4', '--n', '50', '--force'])
if p.returncode: raise SystemExit(f'audit make exited {p.returncode} - reason printed above.')
from pathlib import Path
d = Path(ART) / 'audit_c4'
print()
print('sheet ready:', d / 'audit_pairs.md')
print('1. download audit_pairs.md from the file browser (left panel)')
print('2. for each pair decide: same functionality? label clone / not_clone / unsure')
print('3. fill audit_labels.csv (same file you downloaded the pairs from:', d / 'audit_labels.csv', ')')
print('4. do NOT open audit_key.csv')
print('5. upload the filled audit_labels.csv in section 5.5')

## 5.5 Score the audit — the gate decision

Upload the **filled** `audit_labels.csv` (only that file). Scoring checks the key still matches the triples on disk, then applies the gate: `fn_rate ≤ 0.42` to pass; the pre-registered hope is ≈ 0.

In [ ]:
import json, os, shutil, subprocess, sys
from google.colab import files
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['EMBEDED_AUDIT_DIR'] = ART + '/audit_c4'

uploaded = files.upload()
if 'audit_labels.csv' not in uploaded:
    raise ValueError('Upload the completed file named audit_labels.csv')
shutil.move('audit_labels.csv', os.path.join(ART, 'audit_c4', 'audit_labels.csv'))

p = subprocess.run([sys.executable, '-u', '-m', 'scripts.audit_sample', 'score'])
if p.returncode: raise SystemExit(f'audit score exited {p.returncode} - reason printed above.')

res = json.loads(open(os.path.join(ART, 'audit_c4', 'audit_result.json')).read())
fn = res['C4']['fn_rate']; upper = res['C4']['fn_rate_upper_incl_unsure']
print()
print(f'=== AUDIT GATE: C4 clone rate {fn:.0%} (upper bound incl. unsure {upper:.0%}) ===')
print('    references: C1 ~0-2% | C2 42% [29, 56] | gate: <= 42%, hope ~0%')
if fn > 0.42:
    raise SystemExit('GATE FAILED: C4 contamination is at or above the C2 floor. Training may proceed '
                     'but every C4 comparison is footnoted as "denoising attempted, contamination '
                     'persists" (pre-registration 9.4). Paste this output to the session.')
print('GATE PASSED. C4 comparisons may be made on equal footing once the campaign finishes.')

## 6. The campaign — train + evaluate C4 × seeds 13–18

Six seeds on the SAME frozen triples, giving C4 the same n = 6 as C0–C3 (seeds 13/14/15 re-use the original triple set by design — Flavor A: new runs never overwrite the original `runs/*` tree; C4 seeds 13–18 all write NEW `runs/C4_<seed>` directories).

**Before running:** edit `RUNS` to this VM's slots. Two-VM split: VM1 `[(C4,13),(C4,14),(C4,15)]`, VM2 `[(C4,16),(C4,17),(C4,18)]`. A single VM can do all six (~one VM-day). Interrupting after a slot's push is safe; finished runs print `[cache]`-style skips on re-run — never manually restart a finished run.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from scripts.hf_artifacts import main as hf_main

# ---- Phase 2C4 campaign: six seeds on the frozen v7 triples -------------------
ALL_RUNS = [('C4', s) for s in (13, 14, 15, 16, 17, 18)]

# >>> THIS VM's slots. Edit before running. <<<
RUNS = ALL_RUNS            # <-- EDIT ME (default = this VM would do all six)

print('this VM will run:', RUNS)
assert len(set(RUNS)) == len(RUNS), 'duplicate slots in this VM list'
assert set(RUNS) <= set(ALL_RUNS), 'slots outside the campaign'

# preflight: v7 summary with 'filtered' present, C4 triples on disk
import json
from embeded import settings as S
summary = json.loads((S.ARTIFACTS / 'mining_summary.json').read_text())
run = summary.get('_run') or {}
if run.get('version') != S.VERSION or 'filtered' not in run.get('strategies', []):
    raise SystemExit('stale artifacts: run section 3 (the sanctioned v7 re-mine) first')
if not (S.ARTIFACTS / 'triples_C4.jsonl').exists():
    raise SystemExit('missing triples_C4.jsonl - run section 3 first')

def sh(*args):
    print('$ python -m ' + ' '.join(args), flush=True)
    rc = subprocess.run([sys.executable, '-u', '-m', *args]).returncode
    if rc:
        raise SystemExit(f'`python -m {args[0]}` exited {rc} - reason printed above.')

for condition, seed in RUNS:
    print(f'\n===== TRAIN {condition} seed={seed} =====', flush=True)
    sh('embeded.train', '--condition', condition, '--seed', str(seed))
    print(f'===== EVALUATE {condition} seed={seed} =====', flush=True)
    sh('embeded.evaluate', '--condition', condition, '--seed', str(seed))
    hf_main(['push', '--if-configured'])   # run dir only - NEVER --include-report from a trainer
print('\nCampaign slots done:', RUNS)
print('Re-running this cell is safe: finished runs skip.')

## 7. After this VM's slots finish

Push the run dirs once more (idempotent) and stop — unless this is the **consolidating** VM (last to finish): then also run §7.5.

In [ ]:
from scripts.hf_artifacts import main as hf_main
# Trainer VMs push ARTIFACTS ONLY. The report is rebuilt once, on the consolidating VM.
hf_main(['push', '--if-configured'])
print('run dirs synced. Report/GitHub steps belong to the consolidating VM (next section).')

## 7.5 Consolidating VM only — rebuild the report from the union

Run after every trainer VM has finished and pushed. `--light` is enough (table, audit blocks, figure). The confusion board auto-discovers C0–C4; the results table gains the C4 rows; the C4 audit section was appended to the report by §5.5's score step.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from scripts.hf_artifacts import main as hf_main

hf_main(['pull', '--if-configured', '--light'])

p = subprocess.run([sys.executable, '-u', '-m', 'embeded.evaluate', '--results-table'])
if p.returncode:
    raise SystemExit('--results-table failed - reason printed above.')
print('table now aggregates every discovered run (C0-C3 bands + the new C4 rows).')

import pathlib
from scripts.plot_confusion_matrices import main as cm_main
_runs = pathlib.Path(ART) / 'runs'
_out  = pathlib.Path(ROOT) / 'report' / 'figures' / 'phase2_confusion_matrices.png'
cm_main(['--runs-dir', str(_runs), '--out', str(_out)])

hf_main(['push', '--if-configured', '--include-report'])
print('consolidation pushed. Now the GitHub step (final cell).')

## 8. Push the VM's commits back to GitHub

Same routine as Phase 2V2 §8: commit identity, rebase over the branch, push. Trainers usually have nothing to commit (run dirs live in the HF artifact repo) — the cell is still the safe way to leave the VM.

In [ ]:
import json, os, subprocess, sys, urllib.error, urllib.request

ROOT = '/content/EmbedEd'
REF  = REPO_REF                 # cell 2
GIT_NAME  = ''                  # your GitHub username; '' = derive it from the PAT below
GIT_EMAIL = ''                  # the noreply address shown at https://github.com/settings/emails
COMMIT_DIRTY_REPORT = True      # commit tracked changes under report/ before rebasing
TOKEN_SECRET = 'GITHUB_TOKEN'   # Colab Secrets panel (key icon). No token -> stops before pushing.
API_USER = 'https://api.github.com/user'

def sh(*args, check=False):
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(('GIT_AUTHOR_', 'GIT_COMMITTER_'))}   # these override user.email
    p = subprocess.run(['git', '-C', ROOT, *args], capture_output=True, text=True, env=env)
    shown = (p.stdout + p.stderr).strip()
    print(f"$ git {' '.join(args)} -> exit {p.returncode}")
    if shown:
        print('  ' + shown.replace('\n', '\n  '))
    if check and p.returncode:
        raise SystemExit(f'git {" ".join(args)} failed -- stopping, nothing was lost')
    return p.returncode, p.stdout          # raw: porcelain needs its leading status column

def read_token():                          # Colab Secrets first, then the environment
    try:
        from google.colab import userdata
        value = userdata.get(TOKEN_SECRET)
        if value:
            return value.strip()
    except Exception:
        pass
    return (os.environ.get(TOKEN_SECRET) or '').strip()

# 1. identity first: every commit and every rebase needs one, and a fresh VM has none
if not sh('config', '--get', 'user.email')[1].strip():
    name, email = GIT_NAME, GIT_EMAIL
    token = read_token()
    if not (name and email) and token:
        req = urllib.request.Request(API_USER, headers={'Authorization': f'Bearer {token}',
                                                        'Accept': 'application/vnd.github+json',
                                                        'User-Agent': 'embeded-colab'})
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                info = json.loads(resp.read().decode())
            if info.get('id') and info.get('login'):
                name = info['login']
                email = f"{info['id']}+{info['login']}@users.noreply.github.com"
                print(f"  derived the identity from the PAT: {name} <{email}>")
        except (urllib.error.URLError, urllib.error.HTTPError, OSError, ValueError) as exc:
            print(f'  could not reach {API_USER}: {exc.__class__.__name__}')
    if not (name and email):
        raise SystemExit(
            'No git identity on this VM and nothing to derive one from. Pick one:\n'
            f'  a) set GIT_NAME / GIT_EMAIL at the top of this cell, or\n'
            f'  b) add the PAT as a Colab Secret named {TOKEN_SECRET} (then it is derived above), or\n'
            f'  c) run:  !git -C {ROOT} config user.name "you"  and  '
            f'!git -C {ROOT} config user.email "ID+you@users.noreply.github.com"\n'
            'Nothing was changed; re-run this cell afterwards.')
    sh('config', 'user.name', name, check=True)
    sh('config', 'user.email', email, check=True)
    print('  commits will now be authored by: {} <{}>'.format(
        sh('config', '--get', 'user.name')[1].strip(),
        sh('config', '--get', 'user.email')[1].strip()))

# 2. tracked changes block a rebase; untracked strays usually do not
_, raw = sh('status', '--porcelain=v1')
entries = [(line[:2], line[3:].strip()) for line in raw.splitlines() if line.strip()]
tracked = [path for xy, path in entries if xy.strip() != '??']
untracked = [path for xy, path in entries if xy.strip() == '??']

# 3. commit tracked changes under report/ only; audit_key* is never staged
report = [p for p in tracked if p.startswith('report/') and 'audit_key' not in p]
other = [p for p in tracked if p not in report]
if other:
    raise SystemExit(f'tracked changes outside report/ {other} -- commit or stash them, then re-run')
if tracked:
    if not COMMIT_DIRTY_REPORT:
        raise SystemExit(f'tracked changes {tracked} but COMMIT_DIRTY_REPORT is False')
    sh('add', '--', *report, check=True)
    sh('commit', '-m', 'report: VM measurements + audit separability', check=True)

# 4. public repo: fetch needs no token
sh('fetch', 'origin', REF, check=True)

# 5. an untracked file only blocks the rebase if the remote adds that same path
for path in untracked:
    if sh('ls-tree', '--name-only', 'FETCH_HEAD', '--', path)[1].strip():
        raise SystemExit(f'{path} is untracked here but exists on the remote -- move or delete it, '
                         'then re-run')
    print(f'  (untracked, remote does not have it -> leaving it alone: {path})')

# 6. replay any local commit on top of the branch tip
if sh('rebase', 'FETCH_HEAD')[0]:
    raise SystemExit('rebase stopped -- your commits are safe. Either resolve the conflict and '
                     '`git rebase --continue`, or `git -C %s rebase --abort` to undo it.' % ROOT)
sh('log', '--oneline', '-4')

# 7. scripts.colab_git only exists after the rebase above
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.modules.pop('scripts.colab_git', None)  # a cached pre-rebase copy would shadow the new code
from scripts.colab_git import diagnose, read_secret, sync_and_push

print('\n--- diagnose ---')
for line in diagnose(ROOT, REF):
    print(line)

token = read_secret((TOKEN_SECRET,))
if not token:
    print(f'\nNo {TOKEN_SECRET} in Colab Secrets -> stopping before the push.')
    print('Add the fine-grained PAT as a Colab Secret named GITHUB_TOKEN, then re-run this cell:')
    print('the identity, the commit and the rebase are already done, so it goes straight to push.')
else:
    print('\n--- push ---')
    result = sync_and_push(ROOT, REF, token=token)
    for line in result['lines']:
        print(line)
    print(('ok  ' if result['ok'] else 'FAIL') + f" [{result['stage']}] {result['message']}")


## 9. Reading the results (what changes in the write-up)

Paste back, in this order (each is a designed stop in the session):
1. §3's mining cell output — C4 stats (`jaccard_filtered`, `head_skipped`, `padded`) + the C1–C3-identical assertion line.
2. §4's diagnostics — the C4 row vs C1/C2 (mean cosine, d-gaps).
3. §5.5's gate decision — the `=== AUDIT GATE ===` block.
4. §6's per-slot completions (or any failure/stop output, verbatim).
5. §7.5's table (the C4 rows) + the updated confusion board figure.

The write-up then reports, per pre-registration §9.4: F1(C4) mean ± sd over six seeds against C1 0.7464 ± 0.0176 and C2 0.2478 ± 0.0231; the audit's contamination number beside C2's 42% floor; and which of the three pre-registered outcomes obtained (C4 ≈ C1 / C1 < C4 / C4 ≈ C2). No comparison is headlined unless the §5.5 gate passed.